In [1]:
import sys
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import re
import json

%load_ext autoreload
%autoreload 2
from GeneContent import *

from statsmodels.stats.multitest import multipletests


In [2]:
base_dir = '/Users/cdubin/VMGC_cervical_dysplasia_paper/code/'

In [3]:
sample_metadata = pd.read_csv(f'{base_dir}/cervical_dysplasia/metadata/metadata.csv', index_col=0)
sample_list = sample_metadata.index.tolist()

In [4]:
vmgc_rel_abund = pd.read_csv(f'{base_dir}/cervical_dysplasia/MIDAS3/VMGC_ref/results_C90/merge/species/species_relative_abundance.tsv', sep='\t', index_col=0)
sample_metadata['l_crispatus_rel_abund'] = sample_metadata.index.map(vmgc_rel_abund.loc[988598])


propensity_scores = pd.read_csv(f'{base_dir}/cervical_dysplasia/metadata/propensity_scores.csv', index_col=0)
sample_metadata['propensity_score'] = sample_metadata.index.map(propensity_scores['propensity_score'])
sample_metadata['outcome'] = sample_metadata.index.map(propensity_scores['outcome'])

sample_metadata_for_microslam = sample_metadata.reset_index().rename(columns={'SampleID': 'sample_name'})
sample_metadata_for_microslam.to_csv('l_crispatus_inputs/VMGC_metadata.csv', index=False)


In [5]:
# !rm -r l_crispatus_inputs
# !rm -r l_crispatus_outputs
# !rm -r ./MIDAS_cache

In [6]:
!mkdir l_crispatus_inputs
!mkdir l_crispatus_outputs
!mkdir MIDAS_cache

mkdir: l_crispatus_inputs: File exists
mkdir: l_crispatus_outputs: File exists
mkdir: MIDAS_cache: File exists


In [7]:
results_list = []

paths = {'pangenomes':f'{base_dir}/VMGC/VMGC_db/pangenomes/',
    'sample_metadata':f'{base_dir}/cervical_dysplasia/metadata/metadata.csv'
}

database = 'VMGC'
min_read_count = 250000
sp = 'Lactobacillus crispatus' 
sp_code = 988598

jaccard_threshold = 1
min_mean_depth = 5
min_frequency_threshold = 0.15
max_frequency_threshold = 0.75
min_copy_num = 0.35
centroid_threshold = 75

paths['MIDAS_results'] = f'{base_dir}/cervical_dysplasia/MIDAS3/VMGC_ref/results_C{centroid_threshold}/'

g = GeneContent(sp, species_id=sp_code, outdir='./MIDAS_cache', paths=paths, 
        sample_metadata=sample_metadata, min_copy_num=min_copy_num,
        min_mean_depth=min_mean_depth, 
        orig_centroid_grouping=centroid_threshold,centroid_threshold=centroid_threshold,
        min_frequency_threshold=min_frequency_threshold,max_frequency_threshold=max_frequency_threshold,
        min_sample_read_count=min_read_count,jaccard_threshold=jaccard_threshold,
        use_cache=True, save_cache=True, hide_plots=True)

gene_data = g.centroid_presence_absence_filt
gene_data = gene_data.reset_index().rename(columns={'index':'sample_name'})
outpath = f'l_crispatus_inputs/l_crispatus_VMGC.{g.filter_suffix}.csv'
gene_data.to_csv(outpath, index=False)

Analyzing 352/354 samples with at least 250000 reads
Loaded marker gene profile for 352 samples
Loaded genome-wide depth info for 352 samples
Loaded gene count TSVs for 352 samples
131 samples pass filters: mean_depth >= 5
5058 C75 centroids are detected in at least one sample
702 of 5058 centroids remain after applying 15.0/75.0% min/max frequency filter
Calculating Jaccard distances for 702 centroids
Collapsed 702 centroids into 702 co-abundant groups with Jaccard similarity >= 1


In [8]:

cmd = f"""Rscript run_microSLAM.R --metadata l_crispatus_inputs/VMGC_metadata.csv \
    --gene_data {outpath} \
    --covariate_equation "y ~ propensity_score + l_crispatus_rel_abund + 1" \
    --output_dir l_crispatus_outputs \
    --output_file_suffix {g.filter_suffix} """

!{cmd}

====fit_tau_test::start::==== 
====Fixed-effect coefficients: 

Call:  glm(formula = glm_formula, family = "binomial", data = metadata)

Coefficients:
          (Intercept)       propensity_score  l_crispatus_rel_abund  
              -2.1806                 5.0020                -0.9958  

Degrees of Freedom: 130 Total (i.e. Null);  128 Residual
Null Deviance:	    173.2 
Residual Deviance: 159.7 	AIC: 165.7
====fit_tau_test::initial tau phi:: 1, 1 
====fit_tau_test::final tau phi:: 1, 0 
====fit_tau_test elapsed time: 
   user  system elapsed 
  0.029   0.003   0.033 
====fit_tau_test::end::==== 

Permutation p-value for tau test: 0.59
total time past:0.923 0.034 1.058 0 0


In [9]:
results_path = f'l_crispatus_outputs/microSLAM_{g.filter_suffix}.results.csv'
microslam_results = pd.read_csv(results_path, index_col=0)
if 'glmm_fdr' not in microslam_results.columns:
    print(f'error in FDR calculation: {results_path}, skipping')
    print()
    # continue

print(g.filter_suffix)
sig_fdr = microslam_results[microslam_results['glmm_fdr'] < 0.2]
print(sig_fdr[['beta','glmm_fdr']])

sig_fdr_genes = sig_fdr['glmm_fdr'].to_dict()
results_list += [results_path]
all_sig_genes = []
if len(sig_fdr_genes) > 0:

    cags = pd.read_csv(f'MIDAS_cache/{sp_code}/centroid_groupings.{g.filter_suffix}.csv').groupby('group')['gene'].apply(list)

    for cag in sig_fdr_genes.keys():
        print(cag, sorted(cags[cag]))
        all_sig_genes += sorted(cags[cag])

    print()
    print()

min_copy_num0.35min_read_count250000.mean_depth5.C75.frequency0.15-0.75.jaccardmax1
             beta  glmm_fdr
gene_id                    
CAG_14   1.822820  0.001889
CAG_137  1.306820  0.028833
CAG_648  1.411155  0.062058
CAG_215  1.114829  0.126235
CAG_142  0.910305  0.147280
CAG_656  1.264445  0.147634
CAG_482  0.954488  0.151226
CAG_586  0.954488  0.151226
CAG_530  1.051643  0.153942
CAG_141  1.011564  0.161808
CAG_121  0.874454  0.173849
CAG_211  0.970202  0.183124
CAG_237  0.954419  0.184096
CAG_14 ['MG334.mbin.2_00182']
CAG_137 ['ERR10897979.sbin.2_00762']
CAG_648 ['ERR10897723.sbin.1_00478']
CAG_215 ['ERR10897595.sbin.1_00593']
CAG_142 ['SRR17635712.mbin.7_01059']
CAG_656 ['ERR10898139.sbin.1_00313']
CAG_482 ['ERR10897595.sbin.1_01051']
CAG_586 ['ERR10897583.sbin.2_00776']
CAG_530 ['ERR10897583.sbin.2_00437']
CAG_141 ['ERR4421622.sbin.2_01193']
CAG_121 ['ERR10897595.sbin.1_00591']
CAG_211 ['ERR10897595.sbin.1_00598']
CAG_237 ['ERR10897696.sbin.1_01201']


